# 08 Follow-up 2 (fresh notebook, 2026-11-23): does removing race from the CLIP embeddings (LEACE) close the gap?

Plan written before the run: `docs/FOLLOWUP_LEACE.md`. LEACE (Belrose et al., 2023) is fitted on each fold's training rows and applied to the standardised embeddings; then model A's exact procedure runs. Variants: band (primary: race beyond each group's age-band mix) and plain. Probes: linear + MLP on erased and plain inputs, linear on the head's hidden layer. Exploratory: unadjusted paired CIs, no Holm. GPU optional (T4 estimate 30-60 min; the MLP probes run on CPU). Charts only; no images are opened.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib, json, subprocess, zipfile, hashlib
os.environ['CRF_ENV'] = 'colab'
REPO = pathlib.Path('/content/robust_child_filtering_model')
URL = 'github.com/isaacmutuma/robust_child_filtering_model.git'
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/robust_child_filtering_model')
os.chdir('/content')
!rm -rf {REPO}
# Private repo: add a Colab secret GITHUB_TOKEN (key icon, left bar; fine-grained token,
# read-only Contents access to this one repo).
if subprocess.run(['git', 'clone', '-q', f'https://{URL}', str(REPO)]).returncode != 0:
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN')
    subprocess.run(['git', 'clone', '-q', f'https://x-access-token:{tok}@{URL}', str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'remote', 'set-url', 'origin', f'https://{URL}'], check=True)
!git -C {REPO} log --oneline -1
os.chdir(REPO)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU works, just slower)')

def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()

def done(stem):
    """True if Drive holds a clean manifest for `stem` and its npz still matches it."""
    m = DRIVE_ROOT / 'results' / 'embeddings' / f'{stem}.json'
    if not m.exists(): return False
    d = json.loads(m.read_text())
    npz = DRIVE_ROOT / d['npz']['drive_path']
    return d['git']['dirty'] is False and npz.exists() and sha256(npz) == d['npz']['sha256']

def save_manifest(rel):
    """Copy a results JSON to Drive after checking it came from clean, committed code."""
    src = REPO / 'results' / rel
    d = json.loads(src.read_text())
    assert d['git']['dirty'] is False, 'code was modified in this runtime'
    dst = DRIVE_ROOT / 'results' / rel; dst.parent.mkdir(parents=True, exist_ok=True)
    dst.write_text(src.read_text()); print('saved', rel, 'commit', d['git']['commit'][:7])
    return d

## 1. Preconditions

In [ ]:
A = json.loads((DRIVE_ROOT / 'results/models/A.json').read_text())
assert A['git']['dirty'] is False
assert sha256(DRIVE_ROOT / A['oof_file']['drive_path']) == A['oof_file']['sha256'], "A's OOF file changed"
assert done('utkface__clip_vitl14'), 'UTKFace embeddings missing or changed on Drive'
print(open('docs/FOLLOWUP_LEACE.md').read().split('## Design')[0])
!git -C {REPO} log --oneline -1   # must be 37d1ff9 or later (prediction committed)

## 2. Run both variants

In [ ]:
!python scripts/followup_leace.py
d = save_manifest('followups/LEACE.json')

## 3. Summary

In [ ]:
ra = A['report']
print(f"A: gap {ra['tpr_gap_W_minus_B']['est']:.3f}, MAE {ra['mae_overall']['est']:.2f}, B-W age error {ra['age_err_gap_B_minus_W']['est']:+.2f}")
for n, r in d['variants'].items():
    rep, g, pr = r['report'], r['report']['tpr_gap_W_minus_B'], r['paired_vs_A']
    bg = rep['by_group']
    print(f"\n{n}: gap {g['est']:.3f} [{g['ci'][0]:.3f}, {g['ci'][1]:.3f}], reduction vs A {pr['gap_reduction_A_minus_M']['est']:+.3f} "
          f"[{pr['gap_reduction_A_minus_M']['ci'][0]:+.3f}, {pr['gap_reduction_A_minus_M']['ci'][1]:+.3f}], MAE {rep['mae_overall']['est']:.2f} "
          f"({pr['mae_increase_M_minus_A']['est']:+.2f}), B-W age error {rep['age_err_gap_B_minus_W']['est']:+.2f} {[round(x, 2) for x in rep['age_err_gap_B_minus_W']['ci']]}")
    print('  teen TPR B/W', round(bg['Black']['tpr_teen']['est'], 3), '/', round(bg['White']['tpr_teen']['est'], 3),
          '| max adult FPR', round(max(x['fpr_adult']['est'] for x in bg.values()), 4), '| race removed (linear):', r['race_removed_linear'])
    for p_, v in r['probes']['bal_acc'].items():
        print(f"  probe {p_:14s} {v['est']:.3f} [{v['ci'][0]:.3f}, {v['ci'][1]:.3f}]")
    print('  checks:', [(c['leace_rank'], round(c['max_abs_train_cross_cov_race_onehot_after'], 5), round(c['head_refit_test_mae_diff_vs_oof'], 4)) for c in r['checks']])

## 4. Chart: race leakage before and after erasure (band variant)

In [ ]:
import matplotlib.pyplot as plt
r = d['variants']['band']['probes']['bal_acc']
names = ['plain_linear', 'erased_linear', 'plain_mlp', 'erased_mlp', 'head_z_linear']
est = [r[n]['est'] for n in names]
err = [[r[n]['est'] - r[n]['ci'][0] for n in names], [r[n]['ci'][1] - r[n]['est'] for n in names]]
plt.figure(figsize=(7, 3.5)); plt.bar(range(5), est, yerr=err, capsize=3)
plt.xticks(range(5), names, rotation=20); plt.axhline(0.2, ls='--', c='grey', lw=1); plt.ylabel('race probe balanced accuracy'); plt.ylim(0, 1)
plt.tight_layout(); plt.show()

## 5. Aggregate output for the repo (JSON only)

In [ ]:
out = DRIVE_ROOT / 'step12_LEACE_outputs.zip'
with zipfile.ZipFile(out, 'w') as z:
    z.write(DRIVE_ROOT / 'results/followups/LEACE.json', 'results/followups/LEACE.json')
print(out, out.stat().st_size, 'bytes, sha256', sha256(out))